In [42]:
%pip install -q -U google-genai

Note: you may need to restart the kernel to use updated packages.


In [59]:
import os
import re
import json
import time
from pathlib import Path

from google import genai
from google.genai import types

In [349]:
ROOT = Path.home() / "gemini HTML output"

LANGUAGE = "Assamese"
DOC_NAME = "Lentil_KVK , Thoubal ,Manipur"

WORKDIR_ROOT = ROOT / "Workdir" / LANGUAGE / DOC_NAME
PROMPT_PATH = ROOT / "Prompts" / "page_to_pdf.txt"

MODEL = "gemini-3.1-pro-preview"

START_PAGE = 1
END_PAGE = 2

OVERWRITE_EXISTING = True
MAX_RETRIES = 5
RETRY_WAIT_SECONDS = 15

print("Workdir root:", WORKDIR_ROOT)
print("Prompt path:", PROMPT_PATH)
print("Workdir exists:", WORKDIR_ROOT.exists())
print("Prompt exists:", PROMPT_PATH.exists())
print("API key found:", bool(os.environ.get("GEMINI_API_KEY")))

Workdir root: /home/dhruva/gemini HTML output/Workdir/Assamese/Lentil_KVK , Thoubal ,Manipur
Prompt path: /home/dhruva/gemini HTML output/Prompts/page_to_pdf.txt
Workdir exists: True
Prompt exists: True
API key found: True


In [350]:
GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY")
if not GEMINI_API_KEY:
    raise RuntimeError("GEMINI_API_KEY is not set.")

if not PROMPT_PATH.exists():
    raise FileNotFoundError(f"Prompt file not found: {PROMPT_PATH}")

PROMPT = PROMPT_PATH.read_text(encoding="utf-8").strip()
client = genai.Client(api_key=GEMINI_API_KEY)

print("Prompt loaded")
print("Gemini client ready")

Prompt loaded
Gemini client ready


In [351]:
tools = [
    types.Tool(
        googleSearch=types.GoogleSearch()
    ),
]

generate_content_config = types.GenerateContentConfig(
    thinking_config=types.ThinkingConfig(
        thinking_level="HIGH",
    ),
    tools=tools,
)

In [352]:
def clean_html(text: str) -> str:
    text = text.strip()

    # remove accidental code fences
    text = re.sub(r"^```html\s*", "", text, flags=re.IGNORECASE)
    text = re.sub(r"^```\s*", "", text)
    text = re.sub(r"\s*```$", "", text)

    return text.strip() + "\n"


def translate_page_pdf_to_html(pdf_path: Path, prompt: str) -> str:
    pdf_bytes = pdf_path.read_bytes()
    last_error = None

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            collected = []

            for chunk in client.models.generate_content_stream(
                model=MODEL,
                contents=[
                    types.Content(
                        role="user",
                        parts=[
                            types.Part.from_text(text=prompt),
                            types.Part.from_bytes(
                                data=pdf_bytes,
                                mime_type="application/pdf",
                            ),
                        ],
                    ),
                ],
                config=generate_content_config,
            ):
                if text := chunk.text:
                    collected.append(text)

            full_text = "".join(collected).strip()
            if not full_text:
                raise RuntimeError("Empty response received from Gemini.")

            return clean_html(full_text)

        except Exception as e:
            last_error = e
            print(f"Attempt {attempt} failed for {pdf_path.name}: {e}")
            if attempt < MAX_RETRIES:
                time.sleep(RETRY_WAIT_SECONDS)

    raise RuntimeError(f"Failed after {MAX_RETRIES} attempts for {pdf_path}") from last_error

In [353]:
summary = []

for page_num in range(START_PAGE, END_PAGE + 1):
    page_name = f"page_{page_num:03d}"
    page_dir = WORKDIR_ROOT / page_name
    pdf_path = page_dir / f"{page_name}.pdf"
    html_path = page_dir / "translated.html"
    err_path = page_dir / "translated_error.txt"

    if not pdf_path.exists():
        msg = f"PDF not found: {pdf_path}"
        print(msg)
        summary.append({
            "page": page_name,
            "status": "failed",
            "error": msg,
        })
        continue

    if html_path.exists() and not OVERWRITE_EXISTING:
        print(f"Skipping {page_name}: translated.html already exists")
        summary.append({
            "page": page_name,
            "status": "skipped",
            "reason": "translated.html already exists",
        })
        continue

    print(f"\n=== Processing {page_name} ===")

    try:
        translated_html = translate_page_pdf_to_html(pdf_path, PROMPT)
        html_path.write_text(translated_html, encoding="utf-8")

        summary.append({
            "page": page_name,
            "status": "success",
            "output_file": str(html_path),
        })
        print(f"Saved: {html_path}")

    except Exception as e:
        err = str(e)
        err_path.write_text(err, encoding="utf-8")
        summary.append({
            "page": page_name,
            "status": "failed",
            "error": err,
            "error_file": str(err_path),
        })
        print(f"Failed: {page_name} -> {err}")


=== Processing page_001 ===
Saved: /home/dhruva/gemini HTML output/Workdir/Assamese/Lentil_KVK , Thoubal ,Manipur/page_001/translated.html

=== Processing page_002 ===
Saved: /home/dhruva/gemini HTML output/Workdir/Assamese/Lentil_KVK , Thoubal ,Manipur/page_002/translated.html
